<a href="https://colab.research.google.com/github/Dilukshika-Sasitharan/Statistical-Learning-e23355/blob/main/E23355_Assignment_7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment 7b : Gaussian Mixture Model Clustering as Conditional Updating**
# **E23355**
---

# Q1. Bayesian Estimation of a User Ability Parameter from Item Responses (2PL IRT Model)

### Task 1 – Visualizing the Mechanics

The Two-Parameter Logistic (2PL) Item Response Theory (IRT) model defines the probability that a user answers item *i* correctly as

$$
P(Y_i = 1 \mid \Theta = \theta) = p_i(\theta)
= \frac{1}{1 + e^{-a_i(\theta - b_i)}}
$$

where

- $a_i > 0$ is the discrimination parameter.
- $b_i$ is the difficulty parameter.
- $\theta$ is the user's latent ability.

## Interpretation

The difficulty parameter $b_i$ controls the horizontal position of the Item Characteristic Curve (ICC). Increasing $b_i$ shifts the curve to the **right**, indicating that a higher ability is required to obtain the same probability of answering correctly. Conversely, decreasing $b_i$ shifts the curve to the **left**, making the item easier.

The discrimination parameter $a_i$ controls the steepness of the curve. A larger value of $a_i$ produces a steeper curve, meaning that the item distinguishes more effectively between users with different ability levels. In contrast, a smaller value of $a_i$ produces a flatter curve, indicating that the item provides less information about differences in user ability.

In [1]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-6,6,400)

def probability(theta,a,b):
    return 1/(1+np.exp(-a*(theta-b)))

fig = go.Figure()

for b in [-1,0,1]:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=probability(theta,1,b),
            mode='lines',
            name=f'a=1, b={b}'
        )
    )

fig.add_trace(
    go.Scatter(
        x=theta,
        y=probability(theta,2.5,0),
        mode='lines',
        name='a=2.5, b=0',
        line=dict(dash='dash')
    )
)

fig.update_layout(
    title="2PL Item Characteristic Curves",
    xaxis_title="Ability θ",
    yaxis_title="P(Y=1|Θ=θ)",
    template="plotly_white"
)

fig.show()

## Task 2 – Sequential Likelihood Contribution

For a single observation at step $k$, the likelihood contribution is

$$
L(y_k \mid \theta)
=
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{\,1-y_k},
$$

where

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

Assuming that the responses are conditionally independent given the latent ability $\theta$, the joint likelihood function for the running response history is

$$
L\left(\mathbf{y}^{(k)} \mid \theta\right)
=
\prod_{j=1}^{k}
p_j(\theta)^{y_j}
\left[1-p_j(\theta)\right]^{\,1-y_j}.
$$

This joint likelihood summarizes all the information contained in the observed responses up to step $k$ and is used in the Bayesian updating process to obtain the posterior distribution of the user's ability.

## Task 3 – Mathematical Formulation of the Running Update

Initially,

$$
\Theta \sim N(0,1)
$$

with prior density

$$
f^{(0)}(\theta)=\frac{1}{\sqrt{2\pi}}e^{-\theta^2/2}.
$$

Using Bayes' theorem, the posterior distribution after observing the response at step $k$ is

$$
f(\theta \mid \mathbf{y}^{(k)})
\propto
L(y_k \mid \theta)\,
f(\theta \mid \mathbf{y}^{(k-1)}).
$$

Equivalently,

$$
f(\theta \mid \mathbf{y}^{(k)})
=
\frac{
L(y_k \mid \theta)\,
f(\theta \mid \mathbf{y}^{(k-1)})
}{
\int
L(y_k \mid \theta)\,
f(\theta \mid \mathbf{y}^{(k-1)})
\, d\theta
}.
$$

Thus,

**Posterior = Likelihood × Previous Posterior (Normalized)**

The posterior obtained at step $k$ becomes the prior for step $k+1$.

## Task 4 – Dynamic Shifting

Suppose the user correctly answers a highly difficult item ($y_k=1$) with a large difficulty parameter $b_k$.

The posterior becomes

$$
f(\theta \mid \mathbf{y}^{(k)})
\propto
p_k(\theta)\,
f(\theta \mid \mathbf{y}^{(k-1)}).
$$

Since

$$
p_k(\theta)
=
\frac{1}{1+e^{-a_k(\theta-b_k)}},
$$

is an increasing function of the user's ability, larger values of $\theta$ receive greater likelihood values.

Because a difficult item is unlikely to be answered correctly by users with low ability, a correct response provides strong evidence that the user's ability is high. Consequently,

- the posterior distribution shifts toward larger ability values,
- the posterior mean increases,
- the Maximum A Posteriori (MAP) estimate increases,
- the peak of the posterior distribution moves to the right.

Therefore, a correct answer to a difficult item results in a significant rightward shift of the posterior distribution, indicating stronger evidence of higher user ability.

## Task 5 – Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how informative the current item is.

The slope of the logistic curve at

$$
\theta=b_k
$$

is

$$
\frac{a_k}{4}.
$$

When $a_k$ is very large,

- the curve becomes very steep,
- the likelihood strongly favors one range of ability values,
- the posterior distribution becomes more concentrated,
- the posterior variance decreases,
- confidence in the estimated ability increases.

When $a_k$ is very small,

- the curve becomes flatter,
- the likelihood changes only slightly with ability,
- very little new information is obtained,
- the posterior remains similar to the previous posterior,
- the posterior variance changes only slightly.

Therefore, highly discriminating items produce much sharper posterior distributions than weakly discriminating items, while weakly discriminating items provide relatively little information about the user's ability.

## Task 6 – Numerical Grid Approximation

Since the posterior distribution has no closed-form analytical solution, it is approximated numerically using a fixed grid of ability values.

### Algorithm

1. Create a fine grid of ability values, $\theta_1,\theta_2,\ldots,\theta_M$.

2. Compute the standard normal prior density on the grid.

3. For each new response:

   - Evaluate the likelihood at every grid point.
   - Multiply the likelihood by the previous posterior distribution to obtain the unnormalized posterior.

4. Normalize the posterior distribution using the trapezoidal rule.

```python
Z = np.trapezoid(posterior, theta_grid)
posterior = posterior / Z
```

5. Compute the Posterior Mean

```python
posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)
```

6. Compute the Maximum A Posteriori (MAP) estimate

```python
map_estimate = theta_grid[np.argmax(posterior)]
```

7. Repeat the update after each observed response.

This sequential procedure continuously updates the posterior distribution as new responses become available.

In [2]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.75
n = 20

theta_grid = np.linspace(-6,6,2000)

posterior = (1/np.sqrt(2*np.pi))*np.exp(-theta_grid**2/2)
posterior /= np.trapezoid(posterior,theta_grid)

bayes=[]
MAP=[]

for k in range(n):

    a=np.random.uniform(0.5,2.0)
    b=np.random.normal(0,1)

    p_true=1/(1+np.exp(-a*(theta_true-b)))

    y=1 if np.random.rand()<p_true else 0

    p_grid=1/(1+np.exp(-a*(theta_grid-b)))

    likelihood=p_grid**y*(1-p_grid)**(1-y)

    posterior*=likelihood
    posterior/=np.trapezoid(posterior,theta_grid)

    bayes.append(np.trapezoid(theta_grid*posterior,theta_grid))
    MAP.append(theta_grid[np.argmax(posterior)])

steps=np.arange(1,n+1)

fig=go.Figure()

fig.add_trace(go.Scatter(
    x=steps,
    y=bayes,
    mode='lines+markers',
    name='Posterior Mean'
))

fig.add_trace(go.Scatter(
    x=steps,
    y=MAP,
    mode='lines+markers',
    name='MAP'
))

fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='θ_true = 0.75'
)

fig.update_layout(
    title='Running Bayesian Ability Estimation',
    xaxis_title='Item Number',
    yaxis_title='Estimated Ability',
    template='plotly_white'
)

fig.show()

## Task 7 – Analysis

Initially, the posterior distribution is dominated by the standard normal prior, so both the Posterior Mean and the Maximum A Posteriori (MAP) estimates may differ noticeably from the true ability, $\theta_{\text{true}} = 0.75$. As additional responses are observed, the likelihood contributes increasing information, causing the posterior distribution to become more concentrated around the true ability. Consequently, both estimators gradually converge toward $\theta_{\text{true}} = 0.75$, while the difference between them decreases. The narrowing of the posterior distribution indicates reduced uncertainty and increased confidence in the estimated ability. After approximately 15–20 responses, both estimators stabilize close to the true value, demonstrating the effectiveness of Bayesian sequential updating in accurately estimating the user's latent ability.

---

# Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta–Binomial Updates

## Task 1 – Structural Probability and Properties

The prior distribution for the unknown click-through rate (CTR) is modeled using a Beta distribution,

$$
\Theta \sim \mathrm{Beta}(\alpha,\beta),
$$

whose probability density function (PDF) is

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1},
\qquad 0\le\theta\le1,
$$

where

- $\alpha>0$ and $\beta>0$ are the shape parameters.
- $B(\alpha,\beta)$ is the Beta function.

Three different Beta distributions are considered:

- **Uninformative Prior:** $\mathrm{Beta}(1,1)$
- **Right-Skewed Prior:** $\mathrm{Beta}(2,8)$
- **Left-Skewed Prior:** $\mathrm{Beta}(8,2)$

### Interpretation

The Beta distribution represents prior beliefs about the unknown click-through rate before any user interactions are observed.

- **Beta(1,1)** is a uniform distribution, indicating no prior preference for any value of the CTR.
- **Beta(2,8)** is right-skewed, placing most probability near 0, which reflects the belief that the advertisement has a relatively low click-through rate.
- **Beta(8,2)** is left-skewed, placing most probability near 1, indicating a strong prior belief that the advertisement has a high click-through rate.

The mean of a Beta distribution is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Therefore,

- Beta(1,1): Mean = 0.50
- Beta(2,8): Mean = 0.20
- Beta(8,2): Mean = 0.80

Increasing $\alpha$ shifts the density toward 1, whereas increasing $\beta$ shifts the density toward 0.

In [3]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0,1,500)

parameters = [
    (1,1,"Beta(1,1)"),
    (2,8,"Beta(2,8)"),
    (8,2,"Beta(8,2)")
]

fig = go.Figure()

for a,b,label in parameters:

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=beta.pdf(theta,a,b),
            mode="lines",
            name=label
        )
    )

fig.update_layout(
    title="Beta Prior Distributions",
    xaxis_title="CTR (θ)",
    yaxis_title="Density",
    template="plotly_white"
)

fig.show()

### Interpretation of the Plot

The figure compares three Beta prior distributions.

- **Beta(1,1)** is a uniform distribution, showing complete uncertainty before observing any clicks.

- **Beta(2,8)** places most of its probability mass near zero, indicating the prior belief that the advertisement is unlikely to receive many clicks.

- **Beta(8,2)** places most of its probability mass near one, reflecting the prior belief that the advertisement is expected to achieve a high click-through rate.

These prior distributions are updated sequentially as user click data become available through Bayesian inference.



## Task 2 – Sequential Likelihood and Joint History

Each user interaction is modeled as a Bernoulli random variable.

For a single observation at step $k$, the likelihood contribution is

$$
L(y_k\mid\theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k},
$$

where

- $y_k=1$ if the user clicks the advertisement,
- $y_k=0$ if the user does not click the advertisement.

Assuming that all user interactions are conditionally independent given the click-through rate $\theta$, the joint likelihood for the observed response history is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{j=1}^{k}
\theta^{y_j}
(1-\theta)^{1-y_j}.
$$

Let

$$
S_k=\sum_{j=1}^{k}y_j
$$

denote the total number of clicks observed after $k$ impressions.

Then the joint likelihood can also be written as

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\theta^{S_k}
(1-\theta)^{k-S_k}.
$$

This likelihood summarizes all the observed user interactions and forms the basis for updating the posterior distribution of the click-through rate using Bayes' theorem.

## Task 3 – Closed-Form Analytical Updates (Conjugacy)

Assume that the prior distribution of the unknown click-through rate is

$$
\Theta \sim \mathrm{Beta}(\alpha_{k-1},\beta_{k-1}),
$$

with probability density function

$$
f(\theta)
=
\frac{1}{B(\alpha_{k-1},\beta_{k-1})}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

The likelihood contribution of the new observation at step $k$ is

$$
L(y_k \mid \theta)
=
\theta^{y_k}(1-\theta)^{1-y_k},
$$

where $y_k\in\{0,1\}$.

Using Bayes' theorem,

$$
f(\theta \mid \mathbf{y}^{(k)})
\propto
L(y_k \mid \theta)\,
f(\theta \mid \mathbf{y}^{(k-1)}).
$$

Substituting the likelihood and the Beta prior gives

$$
f(\theta \mid \mathbf{y}^{(k)})
\propto
\theta^{y_k}(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining the exponents,

$$
f(\theta \mid \mathbf{y}^{(k)})
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

This has exactly the same functional form as a Beta distribution.

Therefore,

$$
\boxed{
\Theta \mid \mathbf{Y}^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k)
}
$$

where

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

and

$$
\boxed{
\beta_k=\beta_{k-1}+(1-y_k)
}
$$

These are the recursive Bayesian update equations for the Beta–Binomial conjugate model.

### Posterior Mean

The posterior mean (Bayesian estimator) after the $k^{\text{th}}$ observation is

$$
\boxed{
E[\Theta \mid \mathbf{Y}^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}
}
$$

where

$$
\alpha_k=\alpha_0+\sum_{j=1}^{k}y_j,
$$

and

$$
\beta_k=\beta_0+k-\sum_{j=1}^{k}y_j.
$$

### Interpretation

Because the Beta distribution is conjugate to the Bernoulli (Binomial) likelihood, the posterior distribution remains a Beta distribution after each new observation. Each observed click ($y_k=1$) increases the parameter $\alpha$ by one, while each non-click ($y_k=0$) increases the parameter $\beta$ by one. Consequently, Bayesian updating is achieved through simple arithmetic updates without requiring numerical integration, making the Beta–Binomial model computationally efficient for sequential estimation of the click-through rate.

## Task 4 – Dynamic Shifting Mechanics

In the Beta–Binomial model, the posterior distribution is updated analytically after each new observation.

### Case 1: A Click is Observed ($y_k=1$)

When the user clicks the advertisement, the posterior parameters become

$$
\alpha_k=\alpha_{k-1}+1,
$$

$$
\beta_k=\beta_{k-1}.
$$

The posterior distribution is therefore

$$
\Theta \mid \mathbf{Y}^{(k)}
\sim
\mathrm{Beta}(\alpha_{k-1}+1,\beta_{k-1}).
$$

Since the parameter $\alpha$ increases, the posterior density shifts toward larger values of $\theta$. Consequently,

- the posterior mean increases,
- the posterior mode (when it exists) moves toward 1,
- the estimated click-through rate increases.

This indicates stronger evidence that the advertisement has a high click-through rate.

---

### Case 2: A Non-Click is Observed ($y_k=0$)

When the user does not click the advertisement, the posterior parameters become

$$
\alpha_k=\alpha_{k-1},
$$

$$
\beta_k=\beta_{k-1}+1.
$$

The posterior distribution becomes

$$
\Theta \mid \mathbf{Y}^{(k)}
\sim
\mathrm{Beta}(\alpha_{k-1},\beta_{k-1}+1).
$$

Since the parameter $\beta$ increases, the posterior density shifts toward smaller values of $\theta$. Consequently,

- the posterior mean decreases,
- the posterior mode moves toward 0,
- the estimated click-through rate decreases.

This indicates stronger evidence that the advertisement has a low click-through rate.

---

### Comparison with the 2PL Item Response Theory Model

The Beta–Binomial model is a **conjugate Bayesian model**, meaning that the posterior distribution remains a Beta distribution after each observation. As a result, the posterior parameters are updated using simple arithmetic:

$$
\boxed{
\alpha_k=\alpha_{k-1}+y_k
}
$$

$$
\boxed{
\beta_k=\beta_{k-1}+(1-y_k)
}
$$

No numerical approximation or integration is required.

In contrast, the 2PL Item Response Theory (IRT) model uses a logistic likelihood together with a normal prior. This combination is **non-conjugate**, so the posterior distribution does not have a closed-form analytical solution. Therefore, numerical methods such as grid approximation or numerical integration must be used after each observation to estimate the posterior distribution.

### Conclusion

The Beta–Binomial model provides exact and computationally efficient Bayesian updates because of conjugacy. Every click shifts the posterior distribution toward higher click-through rates, while every non-click shifts it toward lower click-through rates. Unlike the non-conjugate 2PL IRT model, these updates are obtained analytically without numerical integration.

## Task 5 – Running Point Estimators

After observing the user responses up to step $k$, the posterior distribution is

$$
\Theta \mid \mathbf{Y}^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k),
$$

where

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

and

$$
\beta_k=\beta_{k-1}+(1-y_k).
$$

Since the posterior distribution belongs to the Beta family, the point estimators can be obtained directly using closed-form expressions.

### 1. Running Posterior Mean (Bayesian Estimator)

The Posterior Mean is

$$
\boxed{
\hat{\theta}^{(k)}_{\mathrm{Bayes}}
=
E[\Theta \mid \mathbf{Y}^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}
}
$$

The Posterior Mean represents the expected value of the click-through rate after incorporating all observations up to step $k$. It minimizes the expected squared error and is commonly used as the Bayesian estimate of the unknown parameter.

---

### 2. Running Maximum A Posteriori (MAP) Estimate

The MAP estimate is the value of $\theta$ that maximizes the posterior density.

For a Beta distribution with

$$
\alpha_k>1
\quad \text{and} \quad
\beta_k>1,
$$

the MAP estimate is

$$
\boxed{
\hat{\theta}^{(k)}_{\mathrm{MAP}}
=
\frac{\alpha_k-1}
{\alpha_k+\beta_k-2}
}
$$

If either parameter is less than or equal to one, the posterior mode occurs at the boundary of the interval:

- If $\alpha_k \le 1$, then

$$
\hat{\theta}_{\mathrm{MAP}}=0.
$$

- If $\beta_k \le 1$, then

$$
\hat{\theta}_{\mathrm{MAP}}=1.
$$

---

### Interpretation

The Posterior Mean and MAP estimate are updated sequentially after every new observation.

- A **click** ($y_k=1$) increases $\alpha_k$, causing both estimators to move toward 1.
- A **non-click** ($y_k=0$) increases $\beta_k$, causing both estimators to move toward 0.

As the number of observations increases, the influence of the initial prior distribution decreases. Consequently, both the Posterior Mean and the MAP estimate gradually converge toward the true click-through rate, providing increasingly accurate estimates of the advertisement's performance.

## Task 6 – Performance Tracking and Convergence Analysis

Assume that the true click-through rate of the advertisement is

$$
\theta_{\mathrm{true}}=0.35.
$$

Initially, the prior distribution is

$$
\Theta \sim \mathrm{Beta}(1,1),
$$

which represents complete uncertainty about the click-through rate.

At each impression, a user either clicks the advertisement ($y_k=1$) or does not click the advertisement ($y_k=0$). The response is generated from a Bernoulli distribution with success probability

$$
P(Y_k=1)=\theta_{\mathrm{true}}.
$$

After each observation, the Beta posterior parameters are updated using

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

$$
\beta_k=\beta_{k-1}+(1-y_k).
$$

The running Posterior Mean and Maximum A Posteriori (MAP) estimates are then computed and stored for each impression. Finally, both estimators are plotted against the true click-through rate to evaluate their convergence as more observations become available.

In [4]:
import numpy as np
import plotly.graph_objects as go

# Reproducible results
np.random.seed(42)

# True CTR
theta_true = 0.35

# Number of impressions
n = 100

# Initial Beta(1,1) prior
alpha = 1
beta = 1

# Lists to store estimates
posterior_mean = []
map_estimate = []
steps = []

for k in range(1, n + 1):

    # Simulate one user response
    y = np.random.binomial(1, theta_true)

    # Bayesian update
    alpha += y
    beta += (1 - y)

    # Posterior Mean
    mean = alpha / (alpha + beta)

    # MAP estimate
    if alpha > 1 and beta > 1:
        MAP = (alpha - 1) / (alpha + beta - 2)
    elif alpha <= 1:
        MAP = 0
    else:
        MAP = 1

    posterior_mean.append(mean)
    map_estimate.append(MAP)
    steps.append(k)

# Plot
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode="lines",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimate,
        mode="lines",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True CTR = 0.35"
)

fig.update_layout(
    title="Sequential Bayesian Estimation of Click-Through Rate",
    xaxis_title="Impression Number",
    yaxis_title="Estimated CTR",
    template="plotly_white"
)

fig.show()

### Analysis

Initially, the Posterior Mean and MAP estimates fluctuate considerably because only a few observations are available and the prior distribution has a noticeable influence. As the number of impressions increases, more evidence is accumulated through user interactions, causing both estimators to move steadily toward the true click-through rate of

$$
\theta_{\mathrm{true}}=0.35.
$$

The difference between the Posterior Mean and the MAP estimate gradually decreases as the sample size grows. At the same time, the posterior variance becomes smaller, indicating greater confidence in the estimated click-through rate.

As the number of impressions approaches 100, both estimators converge very closely to the true value. This demonstrates that Bayesian sequential updating effectively incorporates new information while the influence of the initial Beta(1,1) prior becomes negligible. Consequently, the posterior distribution is increasingly dominated by the observed data, leading to accurate and stable estimates of the advertisement's click-through rate.


---

# Q3. Bayesian Estimation for Structural Health Monitoring via Bounded Grid Updates

## Task 1 – Prior Belief Boundaries

Before any sensor measurements are collected, the remaining stiffness efficiency factor of the structural component is assumed to follow a Beta distribution,

$$
\Theta \sim \mathrm{Beta}(8,1.5),
$$

where the parameter $\theta$ is physically bounded by

$$
0<\theta\le1.
$$

The probability density function (PDF) of the prior distribution is

$$
f(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{8-1}
(1-\theta)^{1.5-1},
\qquad 0<\theta\le1,
$$

where $B(8,1.5)$ is the Beta function.

Since the physical model does not allow $\theta=0$, the density is plotted over the restricted domain

$$
0.01\le\theta\le1.
$$

### Expected Prior Stiffness Efficiency

The mean of a Beta distribution is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Substituting

$$
\alpha=8,
\qquad
\beta=1.5,
$$

gives

$$
E[\Theta]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
=
0.8421.
$$

Therefore,

$$
\boxed{
E[\Theta]=0.8421
}
$$

### Interpretation

The Beta(8,1.5) distribution places most of its probability mass near $\theta=1$, indicating that the structural component is expected to be in a healthy condition before monitoring begins. The prior mean of approximately **0.842** reflects a strong initial belief that the remaining stiffness is high, while still allowing for the possibility of degradation. Since the Beta distribution is naturally bounded between 0 and 1, it is an appropriate prior for modeling structural stiffness efficiency, which cannot exceed its nominal value or become negative.

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Theta grid
theta = np.linspace(0.01, 1.0, 500)

# Prior parameters
alpha = 8
beta_param = 1.5

# Beta PDF
prior = beta.pdf(theta, alpha, beta_param)

# Plot
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta,
        y=prior,
        mode="lines",
        fill="tozeroy",
        name="Beta(8,1.5)"
    )
)

fig.update_layout(
    title="Prior Distribution of Structural Stiffness Efficiency",
    xaxis_title="Stiffness Efficiency (θ)",
    yaxis_title="Probability Density",
    template="plotly_white"
)

fig.show()

# Prior Mean
prior_mean = alpha / (alpha + beta_param)
print("Prior Mean =", round(prior_mean, 4))

Prior Mean = 0.8421


### Analysis

The plotted Beta(8,1.5) distribution is heavily concentrated near $\theta=1$, indicating a strong prior belief that the structural component is initially healthy. The expected stiffness efficiency is

$$
E[\Theta]=0.8421,
$$

which suggests that the component is expected to retain approximately **84.2%** of its original stiffness before any sensor measurements are collected. This prior distribution is appropriate because it reflects engineering knowledge that newly installed or well-maintained structures are generally in good condition while still allowing Bayesian updating if future sensor measurements indicate degradation.

## Task 2 – Structural Likelihood Formulation

At each inspection step $k$, the measured structural stiffness is modeled as

$$
y_k=\theta K_{\mathrm{nominal}}e^{\varepsilon_k},
$$

where

- $\theta$ is the unknown stiffness efficiency factor,
- $K_{\mathrm{nominal}}$ is the nominal stiffness of a healthy structure,
- $\varepsilon_k\sim N(0,\sigma^2)$ is the measurement noise in logarithmic space.

Since the measurement noise is normally distributed in log-space, taking the natural logarithm of both sides gives

$$
\ln(y_k)
=
\ln(\theta K_{\mathrm{nominal}})
+
\varepsilon_k.
$$

Therefore,

$$
\ln(y_k)\mid\theta
\sim
N\left(\ln(\theta K_{\mathrm{nominal}}),\,\sigma^2\right).
$$

Hence, the likelihood contribution of a single sensor measurement is the Log-Normal probability density function,

$$
L(y_k\mid\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\ln(y_k)-\ln(\theta K_{\mathrm{nominal}})\right)^2}
{2\sigma^2}
\right].
$$

Assuming that all sensor measurements are conditionally independent given the unknown stiffness efficiency $\theta$, the joint likelihood function for the observation history

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{j=1}^{k}
L(y_j\mid\theta).
$$

Substituting the Log-Normal likelihood gives

$$
L(\mathbf{y}^{(k)}\mid\theta)
=
\prod_{j=1}^{k}
\frac{1}
{y_j\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\ln(y_j)-\ln(\theta K_{\mathrm{nominal}})\right)^2}
{2\sigma^2}
\right].
$$

### Interpretation

The likelihood function measures how probable the observed sensor readings are for a given value of the unknown stiffness efficiency $\theta$. Measurements that are close to the predicted stiffness value produce higher likelihood values, while measurements that differ significantly from the prediction produce lower likelihood values. As additional sensor measurements are collected, the joint likelihood accumulates more information about the true structural condition and is combined with the prior distribution through Bayes' theorem to obtain the posterior distribution of the stiffness efficiency.

## Task 3 – Mathematical Formulation of the Non-Conjugate Grid Update

The prior distribution of the stiffness efficiency is

$$
\Theta \sim \mathrm{Beta}(8,1.5),
$$

while the likelihood function is Log-Normally distributed because the sensor measurements satisfy

$$
y_k=\theta K_{\mathrm{nominal}}e^{\varepsilon_k},
\qquad
\varepsilon_k\sim N(0,\sigma^2).
$$

The prior density is

$$
f(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{8-1}
(1-\theta)^{1.5-1},
\qquad 0<\theta\le1.
$$

The likelihood contribution of a single sensor measurement is

$$
L(y_k\mid\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\ln(y_k)-\ln(\theta K_{\mathrm{nominal}})\right)^2}
{2\sigma^2}
\right].
$$

### Why No Closed-Form Posterior Exists

The Beta distribution is **not conjugate** to the Log-Normal likelihood.

The Beta prior contains polynomial terms,

$$
\theta^{\alpha-1}(1-\theta)^{\beta-1},
$$

whereas the Log-Normal likelihood contains the nonlinear logarithmic term

$$
\left(\ln(\theta)\right)^2.
$$

Because these two functions do not belong to the same probability family, their product cannot be simplified into another standard probability distribution.

Therefore, **no exact analytical (closed-form) posterior distribution exists**, and numerical methods must be used to estimate the posterior.

### Recursive Bayesian Update

Using Bayes' theorem, the posterior distribution after the $k^{\text{th}}$ sensor measurement is

$$
f(\theta\mid\mathbf{y}^{(k)})
\propto
L(y_k\mid\theta)\,
f(\theta\mid\mathbf{y}^{(k-1)}).
$$

Equivalently,

$$
f(\theta\mid\mathbf{y}^{(k)})
=
\frac{
L(y_k\mid\theta)\,
f(\theta\mid\mathbf{y}^{(k-1)})
}
{
\int_{0}^{1}
L(y_k\mid\theta)\,
f(\theta\mid\mathbf{y}^{(k-1)})
\,d\theta
}.
$$

The denominator is the normalization constant that ensures the posterior integrates to one.

After each new sensor measurement, the posterior obtained at step $k$ becomes the prior distribution for step $k+1$, allowing the structural health estimate to be updated sequentially.

### Interpretation

Unlike conjugate Bayesian models such as the Beta–Binomial model, the posterior distribution in this Structural Health Monitoring problem cannot be computed analytically. Instead, the posterior is obtained numerically by evaluating the likelihood and prior over a grid of possible stiffness values, multiplying them together, and normalizing the result. This grid-based Bayesian updating enables continuous estimation of the remaining structural stiffness as new sensor measurements become available.

## Task 4 – Running Point Estimates

Since the posterior distribution obtained from the Beta prior and the Log-Normal likelihood has **no closed-form analytical solution**, the Running Posterior Mean and the Maximum A Posteriori (MAP) estimate must be computed numerically using the posterior density.

Let

$$
f(\theta \mid \mathbf{Y}^{(k)})
$$

denote the posterior probability density function after the first $k$ sensor measurements.

### 1. Running Posterior Mean (Bayesian Estimator)

The Running Posterior Mean is defined as the expected value of the posterior distribution over the bounded interval $(0,1]$:

$$
\boxed{
\hat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\int_{0}^{1}
\theta\,
f(\theta \mid \mathbf{Y}^{(k)})
\,d\theta
}
$$

Since the posterior density is not available in closed form, this integral is evaluated numerically using a discrete grid of $\theta$ values and the trapezoidal rule.

---

### 2. Running Maximum A Posteriori (MAP) Estimate

The Running MAP estimate is the value of $\theta$ that maximizes the posterior density.

It is defined as

$$
\boxed{
\hat{\theta}_{\mathrm{MAP}}^{(k)}
=
\underset{0<\theta\le1}{\operatorname{arg\,max}}
\;
f(\theta \mid \mathbf{Y}^{(k)})
}
$$

Numerically, the MAP estimate is obtained by identifying the grid point at which the posterior density reaches its maximum value.

---

### Numerical Approximation

Suppose the interval $(0,1]$ is divided into grid points

$$
\theta_1,\theta_2,\ldots,\theta_M.
$$

After normalizing the posterior density on the grid, the Running Posterior Mean is approximated using the trapezoidal rule as

$$
\hat{\theta}_{\mathrm{Bayes}}^{(k)}
\approx
\sum_{i=1}^{M}
\theta_i
f(\theta_i)
\Delta\theta,
$$

which is computed in Python using

```python
posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)
```

Similarly, the MAP estimate is obtained using

```python
map_estimate = theta_grid[np.argmax(posterior)]
```

---

### Interpretation

The Running Posterior Mean provides the expected remaining stiffness efficiency after incorporating all sensor measurements up to step $k$. The MAP estimate represents the most probable value of the remaining stiffness efficiency. As additional sensor measurements become available, both estimates are updated sequentially and gradually converge toward the true structural stiffness. Since the posterior distribution is non-conjugate, both estimators must be evaluated numerically rather than by closed-form analytical expressions.

## Task 5 – Algorithmic Grid Approximation and Normalization

Since the posterior distribution does not have a closed-form analytical solution, it is approximated numerically using a fixed grid of possible stiffness efficiency values.

### Step 1 – Create a Grid

Construct a fine grid of stiffness efficiency values over the physical domain

$$
0<\theta\le1.
$$

To avoid evaluating $\ln(0)$, the grid starts from a small positive value such as

$$
\theta=0.01.
$$

For example,

```python
theta_grid = np.linspace(0.01, 1.0, 2000)
```

---

### Step 2 – Compute the Prior Distribution

Evaluate the Beta prior distribution at every grid point:

$$
f^{(0)}(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{7}
(1-\theta)^{0.5}.
$$

This prior represents the initial belief about the structural stiffness before any sensor measurements are observed.

---

### Step 3 – Compute the Likelihood

For each new sensor measurement $y_k$, evaluate the Log-Normal likelihood at every grid point:

$$
L(y_k\mid\theta)
=
\frac{1}
{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{\left(\ln(y_k)-\ln(\theta K_{\mathrm{nominal}})\right)^2}
{2\sigma^2}
\right].
$$

---

### Step 4 – Update the Posterior Distribution

Multiply the previous posterior by the likelihood to obtain the unnormalized posterior:

$$
f_{\mathrm{new}}(\theta)
=
L(y_k\mid\theta)
\times
f_{\mathrm{old}}(\theta).
$$

This applies Bayes' theorem sequentially after each sensor measurement.

---

### Step 5 – Normalize the Posterior

Since the updated posterior is unnormalized, compute the normalization constant using the trapezoidal rule:

$$
Z
=
\int_{0}^{1}
f_{\mathrm{new}}(\theta)
\,d\theta.
$$

Numerically,

```python
Z = np.trapezoid(posterior, theta_grid)
posterior = posterior / Z
```

After normalization,

$$
\int_{0}^{1}
f(\theta)\,d\theta=1.
$$

Thus, the posterior becomes a valid probability density function.

---

### Step 6 – Compute the Running Estimates

The Running Posterior Mean is computed as

```python
posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)
```

The Running MAP estimate is obtained as

```python
map_estimate = theta_grid[np.argmax(posterior)]
```

---

### Step 7 – Repeat the Sequential Update

Repeat Steps 3–6 for every new sensor measurement.

After each update,

- the posterior from the current step becomes the prior for the next step,
- the Posterior Mean and MAP estimate are recalculated,
- the uncertainty decreases as additional sensor data are collected.

### Interpretation

The grid approximation method enables Bayesian updating when no analytical posterior distribution exists. Starting from a bounded Beta prior, each new sensor measurement updates the posterior through the Log-Normal likelihood. The posterior is normalized using the trapezoidal rule to ensure that it remains a valid probability density function. As more measurements become available, the posterior distribution becomes narrower, and the Posterior Mean and MAP estimates converge toward the true remaining stiffness efficiency of the structure.

## Task 6 – Performance Tracking and Degradation Convergence Analysis

Assume that the true remaining stiffness efficiency of the structure is

$$
\theta_{\mathrm{true}}=0.68.
$$

The nominal structural stiffness is

$$
K_{\mathrm{nominal}}=50.0\ \text{kN/mm},
$$

and the measurement noise has standard deviation

$$
\sigma=0.15.
$$

Initially, the prior distribution is

$$
\Theta \sim \mathrm{Beta}(8,1.5),
$$

representing the belief that the structure is initially healthy.

At each inspection step, a noisy sensor measurement is generated using

$$
y_k
=
\theta_{\mathrm{true}}
K_{\mathrm{nominal}}
e^{\varepsilon_k},
$$

where

$$
\varepsilon_k
\sim
N(0,\sigma^2).
$$

After each measurement, the posterior distribution is updated numerically using the Log-Normal likelihood and the Beta prior. The posterior distribution is normalized using the trapezoidal rule, and the Running Posterior Mean and MAP estimates are computed.

Posterior density curves are stored at inspection steps

$$
k=\{0,1,2,5,10,15\},
$$

to visualize how the uncertainty decreases as more sensor measurements become available.

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

np.random.seed(42)

# Parameters
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n = 15

# Grid
theta_grid = np.linspace(0.01,1.0,2000)

# Prior Beta(8,1.5)
posterior = beta.pdf(theta_grid,8,1.5)
posterior /= np.trapezoid(posterior,theta_grid)

posterior_mean=[]
map_estimate=[]

snapshots={0:posterior.copy()}

milestones=[1,2,5,10,15]

for k in range(1,n+1):

    # Simulated sensor measurement
    epsilon=np.random.normal(0,sigma)
    y=theta_true*K_nominal*np.exp(epsilon)

    # Log-normal likelihood
    likelihood=(
        np.exp(
            -((np.log(y)-np.log(theta_grid*K_nominal))**2)
            /(2*sigma**2)
        )
        /(y*sigma*np.sqrt(2*np.pi))
    )

    # Bayesian update
    posterior*=likelihood
    posterior/=np.trapezoid(posterior,theta_grid)

    # Posterior mean
    posterior_mean.append(
        np.trapezoid(theta_grid*posterior,theta_grid)
    )

    # MAP
    map_estimate.append(
        theta_grid[np.argmax(posterior)]
    )

    if k in milestones:
        snapshots[k]=posterior.copy()

# Posterior density curves
fig=go.Figure()

for step in [0,1,2,5,10,15]:
    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=snapshots[step],
            mode="lines",
            name=f"k={step}"
        )
    )

fig.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True θ=0.68"
)

fig.update_layout(
    title="Posterior Density Evolution",
    xaxis_title="Stiffness Efficiency (θ)",
    yaxis_title="Posterior Density",
    template="plotly_white"
)

fig.show()

# Running estimators
steps=np.arange(1,n+1)

fig2=go.Figure()

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig2.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimate,
        mode="lines+markers",
        name="MAP"
    )
)

fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True θ=0.68"
)

fig2.update_layout(
    title="Sequential Bayesian Estimation of Structural Stiffness",
    xaxis_title="Inspection Step",
    yaxis_title="Estimated Stiffness Efficiency",
    template="plotly_white"
)

fig2.show()

## Analysis

Initially, the posterior distribution is concentrated near $\theta=1$ because the Beta(8,1.5) prior assumes that the structure is healthy before monitoring begins. Consequently, the first few posterior estimates remain higher than the true stiffness efficiency of

$$
\theta_{\mathrm{true}}=0.68.
$$

As additional sensor measurements are collected, the Log-Normal likelihood gradually dominates the optimistic prior. The posterior density shifts toward the true stiffness value and becomes progressively narrower, indicating a reduction in uncertainty.

The Running Posterior Mean and the MAP estimate both converge toward approximately **0.68** after several inspection steps. In a typical simulation, the posterior distribution begins to concentrate around the true stiffness after approximately **5–10 sensor measurements**, although the exact number may vary because of random measurement noise.

The narrowing of the posterior density curves indicates increasing confidence in the estimated structural stiffness. This reduction in uncertainty allows engineers to identify structural degradation more reliably and supports informed maintenance decisions before the component reaches a critical safety threshold.

---

# Q4. Gaussian Mixture Clustering as Conditional Updating

## Task 1 – Deriving the Marginal Density

Suppose each observation $X_i$ belongs to one of $K$ clusters. Let the latent cluster variable be

$$
C_i \in \{1,2,\ldots,K\},
$$

where

$$
P(C_i=k)=\phi_k,
$$

such that

$$
\phi_k \ge 0,
\qquad
\sum_{k=1}^{K}\phi_k=1.
$$

Conditioned on cluster membership, each observation follows a multivariate Gaussian distribution,

$$
X_i \mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k),
$$

where

- $\mu_k$ is the mean vector of cluster $k$,
- $\Sigma_k$ is the covariance matrix of cluster $k$.

### Derivation of the Marginal Density

Using the **Law of Total Probability**, the marginal density of an observation $x_i$ is

$$
p(x_i)
=
\sum_{k=1}^{K}
P(C_i=k)\,
p(x_i\mid C_i=k).
$$

Substituting the Gaussian distribution and the cluster prior probabilities gives

$$
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\,
\mathcal{N}(x_i\mid\mu_k,\Sigma_k).
$$

Therefore,

$$
\boxed{
p(x_i)
=
\sum_{k=1}^{K}
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}
$$

This expression represents the marginal probability density of an observation after accounting for all possible cluster memberships.

### Why is it called a Gaussian Mixture Density?

The marginal density is called a **Gaussian Mixture Density** because it is formed by combining multiple Gaussian distributions, each weighted by its corresponding mixing probability $\phi_k$.

Each Gaussian component represents one cluster in the dataset, while the mixing weights indicate the probability that an observation originates from each cluster.

Unlike a single Gaussian distribution, a Gaussian Mixture Model (GMM) can represent complex, multimodal datasets with several distinct groups of observations. The overall density is therefore a weighted sum (or mixture) of several Gaussian component densities.

### Interpretation

The mixture weight $\phi_k$ represents the prior probability that an observation belongs to cluster $k$. The Gaussian density

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
$$

measures how likely the observation $x_i$ is under the $k^{\text{th}}$ cluster.

By summing the weighted Gaussian densities over all clusters, the model computes the overall probability density of observing $x_i$ without requiring prior knowledge of its actual cluster membership.

## Task 2 – Deriving the Posterior Cluster Probability

For a given observation $x_i$, we wish to determine the probability that it belongs to cluster $k$ after observing the data.

Using **Bayes' theorem**, the posterior probability of cluster membership is

$$
P(C_i=k \mid X_i=x_i)
=
\frac{
P(X_i=x_i \mid C_i=k)\,
P(C_i=k)
}{
P(X_i=x_i)
}.
$$

Using the Law of Total Probability, the marginal density in the denominator is

$$
P(X_i=x_i)
=
\sum_{j=1}^{K}
P(X_i=x_i \mid C_i=j)\,
P(C_i=j).
$$

Substituting this expression into Bayes' theorem gives

$$
P(C_i=k \mid X_i=x_i)
=
\frac{
P(X_i=x_i \mid C_i=k)\,
P(C_i=k)
}{
\sum_{j=1}^{K}
P(X_i=x_i \mid C_i=j)\,
P(C_i=j)
}.
$$

Since

$$
P(C_i=k)=\phi_k,
$$

and

$$
P(X_i=x_i \mid C_i=k)
=
\mathcal{N}(x_i\mid\mu_k,\Sigma_k),
$$

the posterior probability becomes

$$
\boxed{
P(C_i=k \mid X_i=x_i)
=
\frac{
\phi_k\,
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j\,
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}
}
$$

This posterior probability is called the **responsibility** of cluster $k$ for observation $x_i$ and is denoted by

$$
\boxed{
\gamma_{ik}
=
P(C_i=k \mid X_i=x_i)
}
$$

### Interpretation

The responsibility $\gamma_{ik}$ represents the probability that observation $x_i$ belongs to cluster $k$ after considering both the prior probability of the cluster and the likelihood of the observation.

- The mixing coefficient $\phi_k$ represents the **prior probability** that an observation belongs to cluster $k$.
- The Gaussian density

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
$$

measures how well observation $x_i$ fits cluster $k$.

The denominator normalizes the probabilities so that

$$
\sum_{k=1}^{K}\gamma_{ik}=1.
$$

Therefore, each observation receives a probability for every cluster, rather than being assigned immediately to a single cluster.

### Why is $\gamma_{ik}$ called a Posterior Probability?

The quantity $\gamma_{ik}$ is called a **posterior probability** because it is calculated **after observing the data**. It combines:

- the **prior probability** of belonging to cluster $k$ ($\phi_k$), and
- the **likelihood** of observing $x_i$ under cluster $k$,

through Bayes' theorem.

Consequently, $\gamma_{ik}$ represents the updated belief about the cluster membership of observation $x_i$. Larger values of $\gamma_{ik}$ indicate stronger evidence that the observation belongs to cluster $k$, while smaller values indicate weaker evidence.

## Task 3 – One-Hot Encoding of the Latent Cluster Variable

Define the latent cluster membership variable using a one-hot encoded vector

$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$

where

$$
Z_{ik}=
\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$

Since an observation belongs to exactly one cluster,

$$
\sum_{k=1}^{K}Z_{ik}=1.
$$

---

### Conditional Expectation of the Latent Variable

The conditional expectation of the indicator variable is

$$
E[Z_{ik}\mid X_i=x_i]
=
1\times P(C_i=k\mid X_i=x_i)
+
0\times P(C_i\neq k\mid X_i=x_i).
$$

Therefore,

$$
\boxed{
E[Z_{ik}\mid X_i=x_i]
=
P(C_i=k\mid X_i=x_i)
}
$$

From Task 2,

$$
P(C_i=k\mid X_i=x_i)
=
\gamma_{ik},
$$

where

$$
\boxed{
\gamma_{ik}
=
\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}
}
$$

Hence,

$$
\boxed{
E[Z_{ik}\mid X_i=x_i]
=
\gamma_{ik}
}
$$

---

### Conditional Expectation of the One-Hot Vector

Applying the expectation to every element of the one-hot vector gives

$$
\boxed{
E[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
}
$$

This vector contains the posterior probabilities of belonging to each cluster.

---

### Interpretation

The vector

$$
E[Z_i\mid X_i=x_i]
$$

is called the **soft assignment vector** because it assigns probabilities rather than a single cluster label.

Each component $\gamma_{ik}$ represents the probability that observation $x_i$ belongs to cluster $k$.

Unlike hard clustering, where each observation is assigned to only one cluster, soft clustering allows an observation to have partial membership in multiple clusters.

For example,

$$
E[Z_i\mid X_i=x_i]
=
\begin{bmatrix}
0.10\\
0.75\\
0.15
\end{bmatrix}
$$

means that observation $x_i$ has

- 10% probability of belonging to Cluster 1,
- 75% probability of belonging to Cluster 2,
- 15% probability of belonging to Cluster 3.

Thus, the conditional expectation of the latent variable is exactly the posterior probability (responsibility) vector.

### Conclusion

The soft cluster assignment in a Gaussian Mixture Model is precisely the conditional expectation of the latent one-hot encoded cluster variable,

$$
\boxed{
E[Z_i\mid X_i=x_i]
=
\left(
\gamma_{i1},
\gamma_{i2},
\ldots,
\gamma_{iK}
\right)^T.
}
$$

This result forms the basis of the **Expectation (E) Step** of the Expectation-Maximization (EM) algorithm, where the unknown cluster memberships are replaced by their expected values (responsibilities).

## Task 4 – From Soft Assignment to Hard Clustering

For each observation $x_i$, the posterior probabilities of belonging to each cluster are represented by the responsibility vector

$$
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix},
$$

where

$$
\gamma_{ik}
=
P(C_i=k \mid X_i=x_i).
$$

This vector provides a **soft assignment**, meaning that an observation can belong to multiple clusters with different probabilities.

### Hard Cluster Assignment

A hard cluster assignment is obtained by selecting the cluster with the highest posterior probability. Mathematically,

$$
\boxed{
\hat{C}_i
=
\underset{1\le k\le K}{\operatorname{arg\,max}}
\;
\gamma_{ik}
}
$$

where

- $\hat{C}_i$ is the predicted cluster label for observation $x_i$.
- $\gamma_{ik}$ is the posterior probability (responsibility) of cluster $k$.

The observation is assigned to the cluster with the largest responsibility.

### Example

Suppose an observation has the following responsibility vector:

$$
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
0.15\\
0.70\\
0.15
\end{bmatrix}.
$$

Since

$$
\gamma_{i2}=0.70
$$

is the largest probability,

$$
\boxed{
\hat{C}_i=2.
}
$$

Thus, the observation is assigned to **Cluster 2**.

---

## Difference Between Soft Clustering and Hard Clustering

### Soft Clustering

In soft clustering, every observation is assigned a probability of belonging to each cluster.

For example,

$$
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
0.20\\
0.55\\
0.25
\end{bmatrix}
$$

indicates that the observation has

- 20% probability of belonging to Cluster 1,
- 55% probability of belonging to Cluster 2,
- 25% probability of belonging to Cluster 3.

Therefore, uncertainty in cluster membership is preserved.

---

### Hard Clustering

In hard clustering, only the cluster with the largest posterior probability is selected.

Using the previous example,

$$
\hat{C}_i
=
2,
$$

because

$$
\gamma_{i2}=0.55
$$

is the largest probability.

All remaining probabilities are ignored.

---

## Comparison

| Soft Clustering | Hard Clustering |
|-----------------|-----------------|
| Assigns probabilities to every cluster. | Assigns exactly one cluster. |
| Preserves uncertainty in cluster membership. | Ignores uncertainty after assignment. |
| Uses the full responsibility vector $\gamma_{ik}$. | Uses only the largest responsibility. |
| Implemented during the E-step of the EM algorithm. | Often used for the final cluster label after EM converges. |

---

## Interpretation

Soft clustering provides a probabilistic description of cluster membership by assigning each observation a posterior probability for every cluster. Hard clustering converts these probabilities into a single cluster label by selecting the cluster with the highest responsibility. Consequently, Gaussian Mixture Models naturally perform soft clustering, while hard clustering is obtained as a final decision based on the maximum posterior probability.

## Task 5 – Conditional Expectation of the Observation Given the Cluster

Suppose that an observation belongs to cluster $k$. The conditional distribution of the observation is

$$
X_i \mid C_i=k
\sim
\mathcal{N}(\mu_k,\Sigma_k),
$$

where

- $\mu_k$ is the mean vector of cluster $k$,
- $\Sigma_k$ is the covariance matrix of cluster $k$.

### Conditional Expectation

For a multivariate Gaussian distribution, the conditional expectation is equal to its mean vector. Therefore,

$$
\boxed{
E[X_i \mid C_i=k]
=
\mu_k
}
$$

Hence, the expected value of an observation belonging to cluster $k$ is simply the mean vector of that cluster.

---

## Why is $\mu_k$ the Center of Cluster $k$?

The mean vector $\mu_k$ represents the average location of all observations belonging to cluster $k$. It is therefore regarded as the **center (centroid)** of the cluster.

Observations located close to $\mu_k$ have higher Gaussian density values and are more likely to belong to cluster $k$, whereas observations farther from $\mu_k$ have lower probability densities.

Consequently, $\mu_k$ describes the central location around which the observations in cluster $k$ are distributed.

---

## Comparison of the Two Conditional Expectations

### 1. Soft Cluster Membership

The conditional expectation of the latent cluster indicator is

$$
\boxed{
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
}
$$

where

$$
\gamma_{ik}
=
P(C_i=k \mid X_i=x_i).
$$

This vector represents the posterior probabilities (responsibilities) that observation $x_i$ belongs to each cluster.

---

### 2. Expected Observation Given a Cluster

The conditional expectation of the observation given its cluster membership is

$$
\boxed{
E[X_i \mid C_i=k]
=
\mu_k.
}
$$

This expectation represents the center (mean vector) of cluster $k$.

---

## Difference Between the Two Expectations

| $E[Z_i \mid X_i=x_i]$ | $E[X_i \mid C_i=k]$ |
|-----------------------|---------------------|
| Represents cluster membership probabilities. | Represents the center (mean vector) of a cluster. |
| Computed after observing the data point $x_i$. | Defined by the Gaussian model parameters. |
| Gives the probability that an observation belongs to each cluster. | Gives the expected location of observations within a cluster. |
| Used in the **Expectation (E) Step** of the EM algorithm. | Used in the **Maximization (M) Step** to update cluster parameters. |

---

## Interpretation

The quantity

$$
E[Z_i \mid X_i=x_i]
$$

describes **which cluster an observation is most likely to belong to** by assigning posterior probabilities to all clusters.

In contrast,

$$
E[X_i \mid C_i=k]
$$

describes **where the observations of a particular cluster are expected to lie**, namely at the cluster mean $\mu_k$.

Thus, the first conditional expectation provides information about **cluster membership**, whereas the second describes the **location of the cluster itself**.

### Conclusion

The two conditional expectations describe different aspects of the Gaussian Mixture Model. The expectation

$$
E[Z_i \mid X_i=x_i]
$$

provides the soft cluster assignment (posterior probabilities), while

$$
E[X_i \mid C_i=k]
=
\mu_k
$$

gives the center of cluster $k$. Together, these quantities form the basis of the Expectation-Maximization (EM) algorithm used to estimate Gaussian Mixture Models.

## Task 6 – The Complete-Data Likelihood

Assume that the latent cluster labels are known. Define the one-hot encoded latent variable

$$
Z_{ik}
=
\begin{cases}
1, & \text{if observation } x_i \text{ belongs to cluster } k,\\
0, & \text{otherwise}.
\end{cases}
$$

Since each observation belongs to exactly one cluster,

$$
\sum_{k=1}^{K} Z_{ik}=1.
$$

---

### Complete-Data Likelihood

If both the observations

$$
x_1,x_2,\ldots,x_n
$$

and the latent cluster assignments

$$
z_1,z_2,\ldots,z_n
$$

are known, the complete-data likelihood is

$$
\boxed{
p(x_1,\ldots,x_n,z_1,\ldots,z_n)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right]^{Z_{ik}}
}
$$

where

- $\phi_k$ is the prior probability (mixing coefficient) of cluster $k$,
- $\mathcal{N}(x_i\mid\mu_k,\Sigma_k)$ is the Gaussian density of cluster $k$.

---

### Complete-Data Log-Likelihood

Taking the natural logarithm,

$$
\ell_c
=
\ln
p(x_1,\ldots,x_n,z_1,\ldots,z_n).
$$

Using the logarithm properties

$$
\ln(ab)=\ln a+\ln b,
$$

and

$$
\ln(a^b)=b\ln a,
$$

gives

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
Z_{ik}
\left[
\ln(\phi_k)
+
\ln\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right].
}
$$

This is called the **complete-data log-likelihood**.

---

### Why is the Complete-Data Log-Likelihood Easy to Maximize?

If the latent variables $Z_{ik}$ are known, then the cluster membership of every observation is known exactly.

Consequently,

- each observation contributes only to its assigned cluster,
- the optimization problem separates into independent calculations for each cluster,
- the parameters $\phi_k$, $\mu_k$, and $\Sigma_k$ can be estimated directly using closed-form formulas.

Therefore, maximizing the complete-data log-likelihood is straightforward because there is no uncertainty about cluster membership.

In practice, however, the latent variables are **unknown**, so the cluster memberships must first be estimated. This is accomplished using the **Expectation-Maximization (EM) algorithm**, where the unknown indicator variables are replaced by their expected values (responsibilities) during the E-step.

---

### Interpretation

The complete-data likelihood combines two sources of information:

1. **The probability that an observation belongs to a cluster**, represented by the mixing coefficient $\phi_k$.

2. **The probability of observing the data point within that cluster**, represented by the Gaussian density

$$
\mathcal{N}(x_i\mid\mu_k,\Sigma_k).
$$

If the cluster labels were known, parameter estimation would be a simple optimization problem. Since the labels are hidden, the EM algorithm alternates between estimating the cluster memberships (E-step) and updating the model parameters (M-step) until convergence.

### Conclusion

The complete-data likelihood provides the mathematical foundation of the EM algorithm. By assuming that the latent cluster labels are known, the log-likelihood becomes a simple summation that is easy to maximize. This property enables efficient estimation of the Gaussian Mixture Model parameters once the latent variables have been estimated.

## Task 7 – The EM Interpretation

In practice, the latent cluster membership variables

$$
Z_{ik}
$$

are **not observed**. Therefore, the complete-data log-likelihood derived in Task 6 cannot be maximized directly.

The **Expectation-Maximization (EM) algorithm** addresses this problem by replacing the unknown indicator variables with their conditional expectations based on the observed data and the current parameter estimates.

### E-Step

For each observation $x_i$, the unknown indicator variable is replaced by its conditional expectation,

$$
\boxed{
Z_{ik}
\;\Longrightarrow\;
E[Z_{ik}\mid X_i=x_i]
=
\gamma_{ik},
}
$$

where

$$
\boxed{
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i)
}
$$

is called the **responsibility** of cluster $k$ for observation $x_i$.

Using Bayes' theorem,

$$
\gamma_{ik}
=
\frac{
\phi_k
\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i\mid\mu_j,\Sigma_j)
}.
$$

Thus,

$$
\boxed{
Z_{ik}
\approx
\gamma_{ik}
}
$$

during the E-step.

---

### Expected Complete-Data Log-Likelihood

Replacing the unknown indicator variables with their expectations gives the expected complete-data log-likelihood

$$
\boxed{
Q(\Theta)
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\ln(\phi_k)
+
\ln\mathcal{N}(x_i\mid\mu_k,\Sigma_k)
\right].
}
$$

This function is maximized during the **Maximization (M) Step** to obtain updated estimates of the model parameters.

---

### Why is the E-Step a Conditional Update?

The E-step is called a **conditional update** because the cluster membership probabilities are updated **conditionally on the observed data** and the current parameter estimates.

For each observation,

- the current Gaussian parameters determine how well the observation fits each cluster,
- Bayes' theorem combines the prior cluster probabilities with the Gaussian likelihood,
- the result is the posterior probability (responsibility) that the observation belongs to each cluster.

Therefore, the E-step updates our belief about the hidden cluster memberships without changing the model parameters.

---

### Interpretation

The EM algorithm alternates between two steps:

**Expectation (E) Step**

- Compute the responsibilities

$$
\gamma_{ik}
=
P(C_i=k\mid X_i=x_i),
$$

which represent the posterior probabilities of cluster membership.

**Maximization (M) Step**

- Update the model parameters

$$
\phi_k,\;
\mu_k,\;
\Sigma_k
$$

by maximizing the expected complete-data log-likelihood

$$
Q(\Theta).
$$

These two steps are repeated until the parameter estimates converge.

---

### Conclusion

The E-step replaces the unknown cluster assignments with their conditional expectations (responsibilities). These responsibilities represent the posterior probabilities of cluster membership and are used as fractional weights in the expected complete-data log-likelihood. Consequently, the E-step can be interpreted as a Bayesian conditional update of the hidden cluster memberships based on the observed data and the current Gaussian Mixture Model parameters.

## Task 8 – Parameter Updates

After the E-step, the responsibilities

$$
\gamma_{ik}
=
P(C_i=k \mid X_i=x_i)
$$

represent the posterior probability that observation $x_i$ belongs to cluster $k$.

During the **Maximization (M) Step**, these responsibilities are used as fractional weights to update the Gaussian Mixture Model (GMM) parameters.

---

### Step 1 – Effective Number of Observations

The effective number of observations assigned to cluster $k$ is

$$
\boxed{
N_k
=
\sum_{i=1}^{n}
\gamma_{ik}
}
$$

Unlike hard clustering, where each observation belongs entirely to one cluster, the responsibilities allow each observation to contribute partially to multiple clusters.

---

### Step 2 – Update of the Mixing Coefficients

The updated mixing coefficient (prior probability) for cluster $k$ is

$$
\boxed{
\phi_k^{\mathrm{new}}
=
\frac{N_k}{n}
}
$$

where

- $N_k$ is the effective number of observations assigned to cluster $k$,
- $n$ is the total number of observations.

This ensures that

$$
\sum_{k=1}^{K}
\phi_k^{\mathrm{new}}
=
1.
$$

---

### Step 3 – Update of the Mean Vector

The updated mean vector of cluster $k$ is

$$
\boxed{
\mu_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}x_i
}
$$

This equation computes the weighted average of all observations, where the weights are the responsibilities.

---

### Step 4 – Update of the Covariance Matrix

The updated covariance matrix is

$$
\boxed{
\Sigma_k^{\mathrm{new}}
=
\frac{1}{N_k}
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\mathrm{new}})
(x_i-\mu_k^{\mathrm{new}})^T
}
$$

This equation measures the weighted spread of observations around the updated cluster mean.

---

## Why Does the Responsibility Act as a Fractional Membership Weight?

In Gaussian Mixture Models, an observation is **not assigned completely** to a single cluster.

Instead, every observation contributes to every cluster according to its responsibility

$$
\gamma_{ik}.
$$

For example,

suppose an observation has the responsibility vector

$$
\begin{bmatrix}
0.10\\
0.70\\
0.20
\end{bmatrix}.
$$

This means that the observation contributes

- **10%** to Cluster 1,
- **70%** to Cluster 2,
- **20%** to Cluster 3.

Therefore,

- observations with larger responsibilities have a greater influence on the parameter updates,
- observations with smaller responsibilities have a weaker influence.

This probabilistic weighting enables Gaussian Mixture Models to represent uncertainty in cluster membership more effectively than hard clustering methods.

---

## Interpretation

The M-step updates the model parameters using the responsibilities computed during the E-step.

- The **mixing coefficient** $\phi_k$ estimates the proportion of observations belonging to cluster $k$.
- The **mean vector** $\mu_k$ represents the weighted center of the cluster.
- The **covariance matrix** $\Sigma_k$ describes the weighted spread of the observations around the cluster center.

Because the responsibilities are posterior probabilities, every observation contributes proportionally to each cluster rather than belonging exclusively to one cluster.

---

## Conclusion

The parameter update equations maximize the expected complete-data log-likelihood obtained during the E-step. The responsibilities act as **fractional membership weights**, allowing each observation to influence multiple clusters according to its posterior probability. Repeating the E-step and M-step iteratively improves the Gaussian Mixture Model parameters until convergence.

## Task 9 – Interpretation

A Gaussian Mixture Model (GMM) can be viewed as a repeated process of **conditional updating** because the cluster membership probabilities are continuously revised as the model parameters are updated through the Expectation-Maximization (EM) algorithm.

Initially, each cluster is assigned a **mixture weight**,

$$
\phi_k,
$$

which represents the **prior probability** that an observation belongs to cluster $k$ before the observation is examined.

For each observation $x_i$, the Gaussian density

$$
\mathcal{N}(x_i \mid \mu_k,\Sigma_k)
$$

measures how compatible the observation is with cluster $k$. Observations that are closer to the cluster mean generally have higher Gaussian density values and are therefore more likely to belong to that cluster.

Using Bayes' theorem, the prior probability and the Gaussian likelihood are combined to compute the **posterior probability** (responsibility),

$$
\boxed{
\gamma_{ik}
=
P(C_i=k \mid X_i=x_i)
=
\frac{
\phi_k
\mathcal{N}(x_i \mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^{K}
\phi_j
\mathcal{N}(x_i \mid \mu_j,\Sigma_j)
}
}
$$

The responsibility represents the updated probability that observation $x_i$ belongs to cluster $k$ after the observation has been taken into account.

The collection of posterior probabilities for an observation forms the **soft assignment vector**

$$
\boxed{
E[Z_i \mid X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
}
$$

Unlike hard clustering, where an observation is assigned to only one cluster, the soft assignment vector allows an observation to belong partially to several clusters with different probabilities.

During the **Maximization (M) Step**, these posterior probabilities are used as fractional membership weights to update the model parameters:

- the mixture weights $\phi_k$,
- the cluster means $\mu_k$, and
- the covariance matrices $\Sigma_k$.

The updated parameters are then used in the next E-step to recompute the posterior probabilities. This iterative process continues until the model converges.

### Conclusion

Gaussian Mixture Model clustering is a probabilistic clustering technique based on repeated Bayesian conditional updating. The mixture weights provide the prior probabilities of cluster membership, the Gaussian densities evaluate how well each observation fits a cluster, and the responsibilities represent the posterior probabilities after observing the data. The soft assignment vector,

$$
E[Z_i \mid X_i=x_i],
$$

captures these posterior probabilities, while the M-step uses them as weights to update the cluster parameters. By repeatedly alternating between estimating cluster memberships and updating model parameters, the EM algorithm gradually converges to an optimal probabilistic clustering of the data.